# Did loyalty drive the recovery, and what are the lost young customers worth?

**Context.** A competitor opened across the street from Lululime on 15 March 2024. Revenue has since recovered, and **leadership credits the loyalty program** (launched 1 June 2024) for that recovery (case brief, "The Challenge"). Earlier notebooks in this folder showed the recovery came entirely from customers aged 25+, while 18–24 year-olds never came back.

**Questions this notebook answers**
1. **Did loyalty drive the recovery?** Members' revenue grew, but what does that actually show? Why did older customers recover and young ones didn't, when both had the same program?
2. **What are the lost young customers worth?** How much ARR and profit is missing, and how does winning some back compare with what the store spends on its programs?

### Summary of the answers
- **"Members' revenue grew" is close to automatic.** Customers join in store, so almost everyone who keeps shopping becomes a member: by mid-2025, **90% of revenue came from members**, in every age group.
- **Young customers joined loyalty almost as often (66% vs 70%) but didn't recover.** Loyalty alone can't explain the recovery: the same program, with similar take-up, did nothing for the age group the competitor hit.
- **Where loyalty plausibly helped: older customers came more often.** Customers aged 25+ didn't come back in greater numbers (active customers flat), but they **visited more often: 4.5 → 5.4 visits a year (+22%)**, with slightly smaller baskets. That is the pattern a points program encourages, though we can't fully separate it from other causes.
- **The lost young customers are worth about $550K of ARR and ~$424K of profit a year.** Each young customer brings in ~$430 profit a year. Winning back **half** of them (~$212K profit/year) would equal about half of what redeemed points cost over the last 12 months ($413K).

## Data and definitions

**The data.** Every Lululime transaction from 1 Nov 2022 to 31 Oct 2026, cleaned and joined once in the team's shared code (`src/`):
- `load("lines")`: one row per item line; `load("visits")`: one row per transaction; `load("customers")`: one row per customer; `load("customer_month")`: one row per customer per month, with each customer's ARR.
- `C` holds agreed values, such as `C.PURCHASE` = "Purchase" and `C.LOYALTY_LAUNCH` = 1 June 2024.

**Loyalty rules (from the case organisers, checked in the data):** before 1 June 2024 the program didn't exist. After that everyone is a **Non-member** until their `loyalty_join_date`, and only then earns points. Joining happens in store.

**Terms**
| Term | Meaning |
|---|---|
| Member at the visit | The purchase was made **on or after** the customer joined (tier recorded as Member, Silver or Gold). Not the same as "ever joined" |
| Active customer | Made at least one purchase in the period |
| Mar–Oct window | The longest window available in every year 2023–2026 (the data ends in October 2026). 2023 is the year before the competitor |
| ARR | The team's definition: revenue from repeat customers (visits after their first purchase) over the trailing 12 months, from `customer_month` |
| Profit | Revenue − production cost − repair cost − redeemed points (20 points = $1), as defined in the case |
| 25+ | All customers aged 25 and over, compared with 18–24. "Unknown" age (2% of revenue) is left out |

In [ ]:
import sys
from pathlib import Path

# Make the repo's shared code (src/) importable; this notebook lives in analysis/a_competitor/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.metrics import load      # shared loader for the cleaned tables
from src import config as C       # shared constants: dates, row types, paths

OUT = C.RESULTS_DIR / "a_competitor"     # charts are saved here
OUT.mkdir(parents=True, exist_ok=True)
RED, BLUE, GREY = "#D1495B", "#2B4C7E", "#9AA5B1"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False})

def titles(fig, ax, title, subtitle):
    # Bold takeaway on top, then a smaller grey line saying what the chart shows
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=12)
    ax.set_title(subtitle, loc="left", fontweight="normal", fontsize=9.5, color="#555")

lines = load("lines")
visits = load("visits")
customers = load("customers")
customer_month = load("customer_month")
age = customers.set_index("customer_id").age_band                    # lookup: customer -> age band
join = customers.set_index("customer_id").loyalty_join_date          # lookup: customer -> join date (NaT = never)

p = lines[lines.row_type == C.PURCHASE].copy()                       # purchases only
p["group"] = np.select([p.age_band == "18-24", p.age_band == "Unknown"], ["18-24", "Unknown"], "25+")
p = p[p.group != "Unknown"]
p["member_at_visit"] = p.tier_at_purchase.isin(["Member", "Silver", "Gold"])
GROUPS = ["18-24", "25+"]

def mar_oct(df, year):
    # rows from March to October of one year
    return df[(df.date.dt.year == year) & df.date.dt.month.between(3, 10)]

# Part 1. Did loyalty drive the recovery?

## 1.1 Almost everyone who keeps shopping becomes a member
Share of each month's revenue that came from purchases made by members (at the time of the visit), from the launch on 1 June 2024.

In [ ]:
post = p[p.date >= C.LOYALTY_LAUNCH]
share = (post.groupby(["month", "group"])
             .apply(lambda d: d.loc[d.member_at_visit, "revenue"].sum() / d.revenue.sum(), include_groups=False)
             .unstack())

# Of the customers who bought anything after launch, how many ever joined?
active_after = post.groupby("group").customer_id.unique()
join_rate = pd.Series({g: join.reindex(ids).notna().mean() for g, ids in active_after.items()})

fig, ax = plt.subplots(figsize=(10, 4))
for g, col in [("18-24", RED), ("25+", BLUE)]:
    ax.plot(share.index, share[g], color=col, lw=2.2, label=f"Ages {g}")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylim=(0, 1), ylabel="Share of sales made by members")
titles(fig, ax, "Almost everyone who keeps shopping joins the loyalty program, young and old alike",
       "Share of each month's sales made by loyalty members, from the program's launch (June 2024)")
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(OUT / "a4_member_share.png", bbox_inches="tight"); plt.show()

print("Share of customers active after launch who joined:", ", ".join(f"{g} {join_rate[g]:.0%}" for g in GROUPS))
print("Member share of revenue, first month / mid-2025:",
      ", ".join(f"{g} {share[g].iloc[0]:.0%} / {share.loc['2025-06-01', g]:.0%}" for g in GROUPS))

**What this shows.**
- **People joined straight away:** in the first month, members already made over half of all sales; a year later, about 9 out of every 10 dollars.
- **Young shoppers joined almost as often as older ones:** 66% vs 70% of everyone who shopped after the launch.

**Why it matters:** if nearly everyone who shops joins, then "members' sales went up" just means "the people who kept shopping kept shopping". It doesn't prove the program *made* anyone shop more. Leadership's evidence for crediting loyalty is weaker than it looks.

## 1.2 Same program, very different outcomes
If loyalty drove the recovery, it should have helped every age group that joined. Revenue in Mar–Oct each year, indexed to Mar–Oct 2023 = 100:

In [ ]:
rev = pd.DataFrame({y: mar_oct(p, y).groupby("group").revenue.sum() for y in [2023, 2024, 2025, 2026]})
rev_idx = rev.div(rev[2023], axis=0) * 100
(rev_idx.round(0).style.format("{:.0f}")
 .set_caption("<b>Mar–Oct revenue, 2023 = 100</b>. Both groups had the same loyalty program and similar join rates"))

**What this shows.** Shoppers aged 25+ dipped to 90 in 2024, then rose to 112 (2025) and 123 (2026): they now buy more than before the competitor. 18–24 year-olds stayed at about half (53–58).

**Why it matters:** both groups got the same loyalty program and joined it at similar rates, yet only one recovered. So the loyalty program can't be what drove the recovery; it did nothing to bring young shoppers back.

## 1.3 How did customers aged 25+ recover?
Revenue = **active customers × visits per customer × spend per visit** (Mar–Oct of each year; a visit = a transaction with a purchase).

In [ ]:
def drivers(df, year):
    w = mar_oct(df, year)
    g = w.groupby("group").agg(customers=("customer_id", "nunique"), visits=("transaction_id", "nunique"),
                               revenue=("revenue", "sum"))
    g["visits per customer"] = g.visits / g.customers
    g["spend per visit"] = g.revenue / g.visits
    return g[["customers", "visits per customer", "spend per visit", "revenue"]]

d = {y: drivers(p, y) for y in [2023, 2024, 2025, 2026]}
drv = pd.concat(d, names=["year"]).swaplevel().sort_index()
display(drv.style.format({"customers": "{:,.0f}", "visits per customer": "{:.2f}",
                          "spend per visit": "${:,.0f}", "revenue": "${:,.0f}"})
        .set_caption("<b>Revenue drivers, Mar–Oct each year</b>"))

chg = pd.DataFrame({g: (d[2025].loc[g] / d[2023].loc[g] - 1) for g in GROUPS}).loc[["customers", "visits per customer", "spend per visit"]]
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(3); w = 0.36
for i, (g, col) in enumerate([("18-24", RED), ("25+", BLUE)]):
    vals = chg[g].values
    ax.bar(x + (i - 0.5) * w, vals, w, color=col, label=f"Ages {g}")
    for xi, val in zip(x + (i - 0.5) * w, vals):
        ax.text(xi, val + (0.012 if val >= 0 else -0.012), f"{val:+.0%}", ha="center",
                va="bottom" if val >= 0 else "top", fontsize=9)
ax.axhline(0, color="#333", lw=0.8)
ax.set_xticks(x, ["Number of shoppers", "Shopping trips per shopper", "Spend per trip"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylabel="Change, 2025 vs. 2023", ylim=(-0.55, 0.35))
titles(fig, ax, "Older shoppers bounced back by coming more often; young shoppers simply stopped coming",
       "March to October 2025 compared with the same months of 2023 (before the competitor)")
ax.legend(loc="upper left")
plt.tight_layout(); fig.savefig(OUT / "a4_recovery_drivers.png", bbox_inches="tight"); plt.show()

**What this shows (2025 compared with 2023).**
- **Shoppers aged 25+:** about the same number of people (−4%), but each came **22% more often**, spending a little less per trip (−4%). By 2026 they came even more often: about 6 trips between March and October, up from 4.5.
- **Shoppers aged 18–24:** 43% fewer people. Those who stayed came about as often as before.

**Why it matters:** "come back more often, spend a bit less each time" is exactly what a points program encourages. So loyalty **probably did help older customers return more often**, and that's where the recovery came from. We can't be fully sure, because older shoppers losing interest in the new competitor would look the same.

## 1.4 The same customers, before and after
Customers who bought in Mar–Oct 2023 (before the competitor and before loyalty existed), split by whether they had joined loyalty by October 2025. How many were still buying in Mar–Oct 2025, and how did those customers' spending change?

In [ ]:
rows = []
for g in GROUPS:
    r23 = mar_oct(p[p.group == g], 2023).groupby("customer_id").revenue.sum()
    r25 = mar_oct(p[p.group == g], 2025).groupby("customer_id").revenue.sum().reindex(r23.index).fillna(0)
    joined = join.reindex(r23.index).le(pd.Timestamp("2025-10-31")).values   # NaT (never joined) -> False
    for label, mask in [("joined loyalty", joined), ("never joined", ~joined)]:
        a, b = r23[mask], r25[mask]
        still = b > 0
        rows.append({"age group": g, "group": label, "customers (2023)": int(mask.sum()),
                     "still buying in 2025": still.mean(),
                     "2023 spend (still buying)": a[still].mean(), "2025 spend (still buying)": b[still].mean(),
                     "change (still buying)": b[still].sum() / a[still].sum() - 1})
same = pd.DataFrame(rows).set_index(["age group", "group"])
(same.style.format({"customers (2023)": "{:,}", "still buying in 2025": "{:.0%}", "2023 spend (still buying)": "${:,.0f}",
                    "2025 spend (still buying)": "${:,.0f}", "change (still buying)": "{:+.0%}"})
 .set_caption("<b>Customers active in Mar–Oct 2023: did they stay, and did they spend more?</b>"))

**What this shows.**
- **The people who joined were already the best customers before loyalty existed.** In 2023, future members aged 25+ spent about $1,130 each between March and October; people who never joined spent about $310. Good customers join, so members look good partly because of who they are.
- **Older members who kept shopping spent 27% more in 2025 than in 2023.** That fits the idea that loyalty encouraged them, though some of it is simply that the keenest shoppers both join and stay.
- **Young members who kept shopping spent about the same as before (−4%).** Loyalty didn't get young people to spend more either.
- **Almost nobody who never joined was still shopping** (6–8%). "Non-member" now mostly means "someone who stopped coming".

# Part 2. What are the lost young customers worth?

## 2.1 ARR by age band: before the competitor vs now
ARR (the team's definition) at the end of **February 2024**, the last month before the competitor, vs **October 2026**, the latest month.

In [ ]:
cm = customer_month.assign(age=customer_month.customer_id.map(age))
arr = cm.groupby(["month", "age"]).arr_12m.sum().unstack()
before, now = arr.loc["2024-02-01"], arr.loc["2026-10-01"]
arr_tbl = pd.DataFrame({"ARR Feb 2024": before, "ARR Oct 2026": now, "change ($)": now - before, "change (%)": now / before - 1})
arr_tbl.loc["Total"] = [before.sum(), now.sum(), now.sum() - before.sum(), now.sum() / before.sum() - 1]
display(arr_tbl.style.format({"ARR Feb 2024": "${:,.0f}", "ARR Oct 2026": "${:,.0f}", "change ($)": "${:+,.0f}", "change (%)": "{:+.0%}"})
        .set_caption("<b>ARR by age band</b>: last month before the competitor vs latest month"))

ch = (now - before).drop("Unknown").sort_values()
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.barh(ch.index, ch.values / 1e3, color=[RED if v < 0 else BLUE for v in ch.values])
for i, v in enumerate(ch.values):
    ax.text(v / 1e3 + (8 if v >= 0 else -8), i, f"{'+' if v >= 0 else '−'}${abs(v)/1e3:,.0f}K", va="center",
            ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color="#333", lw=0.8)
ax.set(xlabel="Change in yearly sales from repeat shoppers ($ thousands)", xlim=(-750, 600))
titles(fig, ax, "Young shoppers now bring in $549K less a year; older shoppers more than made up for it",
       "Yearly sales from repeat shoppers (the case's ARR), just before the competitor (Feb 2024) vs now (Oct 2026)")
plt.tight_layout(); fig.savefig(OUT / "a4_arr_change_by_age.png", bbox_inches="tight"); plt.show()

**What this shows.** In total, yearly sales from repeat shoppers grew from $6.29M to $6.67M (+6%), which looks healthy. But it hides two opposite moves: **young shoppers now bring in about $550K less a year (−46%)**, while every older age group brings in more.

**Why it matters:** the store relies more and more on older customers. If they ever drift away too, there are far fewer young customers coming up behind them.

## 2.2 What one young customer is worth
Revenue and profit from 18–24 customers in the **year before the competitor** (Mar 2023 – Feb 2024) vs the **latest 12 months** (Nov 2025 – Oct 2026). Profit uses the case's definition, including redeemed points.

In [ ]:
v = visits.assign(age=visits.customer_id.map(age))
def year_value(start, end, group="18-24"):
    w = v[(v.date >= start) & (v.date < end) & (v.age == group)]
    buyers = w.loc[w.has_purchase, "customer_id"].nunique()
    return pd.Series({"revenue": w.revenue.sum(), "profit": w.profit.sum(), "buyers": buyers,
                      "profit per buyer": w.profit.sum() / buyers})
val = pd.DataFrame({"Year before competitor (Mar 23 – Feb 24)": year_value("2023-03-01", "2024-03-01"),
                    "Latest 12 months (Nov 25 – Oct 26)": year_value("2025-11-01", "2026-11-01")}).T
val.loc["Change"] = val.iloc[1] - val.iloc[0]
(val.style.format({"revenue": "${:,.0f}", "profit": "${:,.0f}", "buyers": "{:,.0f}", "profit per buyer": "${:,.0f}"})
 .set_caption("<b>18–24 customers: yearly value then and now</b>"))

**What this shows.** Before the competitor, young shoppers brought in **$1.27M of sales and $804K of profit a year**. Now it's $678K and $380K: **$424K less profit every year**, because about 900 fewer young people shop here. Each young shopper is still worth about **$430 of profit a year**. The ones who stayed aren't worth less; there are just far fewer of them.

## 2.3 What would winning them back be worth?
Scenarios: Lululime wins back 25%, 50% or 100% of the lost young customers. Compared with what its programs cost over the **latest 12 months** (Nov 2025 – Oct 2026).

In [ ]:
lost_profit = -val.loc["Change", "profit"]
lost_arr = -(now - before)["18-24"]
last12 = visits[(visits.date >= "2025-11-01") & (visits.date < "2026-11-01")]
points_cost, repair_cost = last12.redeemed_dollars.sum(), last12.repair_cost.sum()

scen = pd.DataFrame({f"Win back {int(s*100)}%": {"young customers regained": -val.loc["Change", "buyers"] * s,
                                                  "ARR regained": lost_arr * s, "profit regained / year": lost_profit * s}
                     for s in [0.25, 0.5, 1.0]}).T
display(scen.style.format({"young customers regained": "{:,.0f}", "ARR regained": "${:,.0f}", "profit regained / year": "${:,.0f}"})
        .set_caption("<b>Value of winning back lost 18–24 customers</b>"))
print(f"For comparison, latest 12 months: redeemed points cost ${points_cost:,.0f}; repairs cost ${repair_cost:,.0f}")

fig, ax = plt.subplots(figsize=(9, 3.8))
labels = list(scen.index) + ["Points redeemed\n(latest 12 months)", "Repairs\n(latest 12 months)"]
values = list(scen["profit regained / year"]) + [points_cost, repair_cost]
colors = [BLUE] * 3 + [GREY, GREY]
ax.bar(labels, np.array(values) / 1e3, color=colors)
for i, val_ in enumerate(values):
    ax.text(i, val_ / 1e3 + 8, f"${val_/1e3:,.0f}K", ha="center", fontsize=9)
ax.set(ylabel="$ thousands per year", ylim=(0, max(values) / 1e3 * 1.18))
titles(fig, ax, "Winning back half the young shoppers we lost would add ~$212K profit a year",
       "Blue: extra profit per year if Lululime wins back 25%, 50% or all of them. Grey: what points and repairs cost in the last 12 months")
ax.tick_params(axis="x", labelsize=8.8)
plt.tight_layout(); fig.savefig(OUT / "a4_win_back_value.png", bbox_inches="tight"); plt.show()

**What this shows.**
- **Winning back half the young shoppers we lost** (about 460 people) would add **about $212K of profit a year** (and ~$275K of yearly repeat sales).
- For scale: that's about **half of what loyalty points cost** over the last 12 months ($413K), and more than the whole repair program ($134K).
- **Rule of thumb for a budget:** each young shopper won back is worth about $430 of profit a year. Any offer that costs less than that per person kept pays for itself within a year.

**Caveat:** these are simple scenarios, not forecasts. They assume regained customers behave like today's young customers, and they don't include the cost of whatever offer wins them back.

## Key numbers

In [ ]:
key = {
    "Share of revenue from members, mid-2025 (18-24 / 25+)": f"{share.loc['2025-06-01', '18-24']:.0%} / {share.loc['2025-06-01', '25+']:.0%}",
    "Active customers who joined loyalty (18-24 / 25+)": f"{join_rate['18-24']:.0%} / {join_rate['25+']:.0%}",
    "Mar–Oct revenue index 2026, 2023 = 100 (18-24 / 25+)": f"{rev_idx.loc['18-24', 2026]:.0f} / {rev_idx.loc['25+', 2026]:.0f}",
    "25+ visits per customer, Mar–Oct 2023 → 2025": f"{d[2023].loc['25+', 'visits per customer']:.1f} → {d[2025].loc['25+', 'visits per customer']:.1f}",
    "25+ active customers, 2025 vs 2023": f"{chg.loc['customers', '25+']:+.0%}",
    "18-24 ARR, Feb 2024 → Oct 2026": f"${before['18-24']:,.0f} → ${now['18-24']:,.0f} ({now['18-24'] / before['18-24'] - 1:+.0%})",
    "Total ARR, Feb 2024 → Oct 2026": f"${before.sum():,.0f} → ${now.sum():,.0f}",
    "18-24 profit lost per year": f"${lost_profit:,.0f}",
    "Profit per young buyer per year": f"${val.iloc[1]['profit per buyer']:,.0f}",
    "Win back 50%: ARR / profit per year": f"${scen.iloc[1]['ARR regained']:,.0f} / ${scen.iloc[1]['profit regained / year']:,.0f}",
    "Latest 12 months: points cost / repair cost": f"${points_cost:,.0f} / ${repair_cost:,.0f}",
}
pd.Series(key, name="value").to_frame()

## Conclusions and caveats
1. **Leadership's evidence for loyalty is weaker than it looks.** Almost every active shopper joined, so member revenue growth mostly reflects who kept shopping.
2. **Loyalty didn't bring back the customers the competitor took.** Young customers joined at almost the same rate but stayed ~45% down.
3. **Loyalty plausibly helped older customers visit more often** (+22% visits per customer), which is where the recovery came from. This is the strongest case *for* the program, and it doesn't depend on expanding perks further.
4. **The young segment is worth fighting for:** ~$550K of ARR and ~$424K of profit a year are missing. Winning back half would be worth about half the current points bill.

**What this means for leadership's plan.** Expanding perks rewards customers who already shop at Lululime. The biggest gap is customers who stopped coming, particularly young ones. Part of the loyalty budget would be better spent on targeted win-back offers for 18–24s (e.g. on staples like leggings, where the competitor undercut prices).

**Caveats.** Joining loyalty is a choice made by engaged customers, so member vs non-member comparisons are biased toward members. The visit increase among older customers could also reflect the competitor's novelty fading. Scenario values assume regained customers behave like current young customers and exclude the cost of winning them back.

---
*Team reference: Stream A extension (angles 1 and 3). Findings F11–F12 in `results/a_competitor/findings.md`. Charts saved to `results/a_competitor/`. Relevant to slides 5 (loyalty) and 10 (recommendation).*